In [ ]:
# importing the required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 5)

In [ ]:
# importing the dataset from kaggle path /kaggle/input/competitions/midad-ml-competition/train_set_.csv
data = pd.read_csv("/kaggle/input/competitions/midad-ml-competition/train_set_.csv",sep=';')

# checking the first 5 rows of the dataset
data.head()

# checking the shape of the dataset
data.shape

# checking the info of the dataset
data.info()

# checking the statistical summary of the dataset
data.describe()

In [ ]:
data=data.drop(columns=["ID"], axis=1)
data.head()

### preprocessing the dataset

In [ ]:
# checking for missing values
data.isnull().sum()

In [ ]:
# checking for duplicate values
data.duplicated().sum()

In [ ]:
# present the data with filter feuling type = 'Electric'
data[data['Fuel_Type'] == 'Electric']

we see the 2 missing values in Mileage come with Electric Fuel Tupe

In [ ]:
kmkgCNG = 0
kmkgpetrol = 0
kmkgDiesel = 0
kmkgLPG = 0
kmplCNG = 0
kmplPetrol = 0
kmplDiesel = 0
kmplLPG = 0

for i , j in zip(data.Mileage , data.Fuel_Type):
    if str(i).endswith("km/kg") and j == "CNG":
        kmkgCNG+=1
    elif str(i).endswith("km/kg") and j == "Petrol":
        kmkgpetrol+=1
    elif str(i).endswith("km/kg") and j == "Diesel":
        kmkgDiesel+=1
    elif str(i).endswith("km/kg") and j == "LPG":
        kmkgLPG+=1
    elif str(i).endswith("kmpl") and j == "CNG":
        kmplCNG+=1
    elif str(i).endswith("kmpl") and j == "Petrol":
        kmplPetrol+=1
    elif str(i).endswith("kmpl") and j == "Diesel":
        kmplDiesel+=1
    elif str(i).endswith("kmpl") and j == "LPG":
        kmplLPG+=1
print('The number of rows with Km/Kg and CNG : {} '.format(kmkgCNG))
print('The number of rows with Km/Kg and Petrol : {} '.format(kmkgpetrol))
print('The number of rows with Km/Kg and Diesel : {} '.format(kmkgDiesel))
print('The number of rows with Km/Kg and LPG : {} '.format(kmkgLPG))
print('The number of rows with kmpl and CNG : {} '.format(kmplCNG))
print('The number of rows with Km/L and Petrol : {} '.format(kmplPetrol))
print('The number of rows with Km/L and Diesel : {} '.format(kmplDiesel))
print('The number of rows with Km/L and LPG : {} '.format(kmplLPG))
print('The total number of rows : {} '.format(kmkgCNG + kmkgpetrol + kmkgDiesel + kmkgLPG + kmplCNG + kmplPetrol + kmplDiesel + kmplLPG))


### we have Km/Kg unit with CNG and LPG fuel type and it's so fewer than kmpl
so we should convert Km/Kg to Km/L and after that we use Mileage's column to float

1 kg of CNG occupies roughly 1.4 to 1.5 Liters of volume equivalent under normal dispenser conditions.

so our formula will be:

###     1 KmpL = 1 KmpKg / 1.4 (CNG)

1 Kg of LPG occupies approximately 1.84 Liters of volume under standard automotive fuel tank pressures.

so our formula will be:

###     1 KmpL = 1 KmpKg / 1.84 (LPG)

In [ ]:
# 1. Clean the string text and extract raw numerical values
def convert_mileage(row):
    mileage_str = str(row["Mileage"]).strip().lower()

    # Handle missing, null, or invalid entries safely
    if (
        mileage_str == "nan"
        or mileage_str == ""
        or mileage_str == "null"
    ):
        return np.nan

    # Extract numerical part from strings like "26.6 km/kg" or "19.67 kmpl"
    try:
        numeric_value = float(mileage_str.split()[0])
    except (ValueError, IndexError):
        return np.nan

    # 2. Apply fuel-specific logic
    fuel_type = str(row["Fuel_Type"]).strip().upper()

    if fuel_type == "CNG":
        # Convert km/kg to kmpl baseline by dividing by 1.4
        return round(numeric_value / 1.4, 2)
    
    elif fuel_type == "LPG":
        # Convert km/kg to kmpl baseline by dividing by 1.84
        return round(numeric_value / 1.84, 2)

    elif fuel_type in ["PETROL", "DIESEL"]:
        # Petrol and Diesel are already natively in kmpl
        return round(numeric_value, 2)

    return numeric_value

In [ ]:
data["Mileage"] = data.apply(convert_mileage, axis=1)
data

we remember there is no Mileage value with electric fuel type
so we should fill it with 0

In [ ]:
# filling missing values for Mileage with Electric fuel tupe by 0
data.loc[(data['Fuel_Type'] == 'Electric') & (data['Mileage'].isna()), 'Mileage'] = 0